In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
def newest(pattern, label):
    paths = list(INPUT.rglob(pattern))
    if not paths:
        raise FileNotFoundError(f"Attach the notebook output containing {label} ({pattern}).")
    return max(paths, key=lambda p: p.stat().st_mtime)

nuts_path = newest("day9_7_nuts_case_diagnostics.csv", "Day 9.7 NUTS diagnostics")
interval_path = newest("day9_7_posterior_recovery_intervals.csv", "Day 9.7 recovery intervals")
map_path = newest("day9_7_map_case_summary.csv", "Day 9.7 MAP case summary")
alias_path = newest("day9_3_binary_alias_candidate_scores.csv", "Day 9.3 BIC alias candidates")

nuts = pd.read_csv(nuts_path)
intervals = pd.read_csv(interval_path)
map_cases = pd.read_csv(map_path)
aliases = pd.read_csv(alias_path)
for name, df, cols in [
    ("NUTS diagnostics", nuts, {"simulation_row", "status", "max_r_hat", "min_effective_sample_size", "divergences"}),
    ("posterior intervals", intervals, {"simulation_row", "parameter", "truth_inside_90pct_interval"}),
    ("MAP summary", map_cases, {"simulation_row", "map_period_days", "known_period_days_for_evaluation", "period_recovered_within_tolerance"}),
    ("alias candidates", aliases, {"simulation_row", "bic", "fit_period_days"}),
]:
    if not cols.issubset(df.columns):
        raise ValueError(f"{name} missing columns: {sorted(cols-set(df.columns))}")

OUT = WORK / ("day9-8-12case-audit-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUT.mkdir(parents=True, exist_ok=False)
print("[PASS] Day 9.7 diagnostics:", nuts_path)
print("[PASS] Day 9.7 intervals:", interval_path)
print("[PASS] Day 9.7 MAP table:", map_path)
print("[PASS] Day 9.3 candidate scores:", alias_path)


In [ ]:
status = nuts[["simulation_row", "status", "max_r_hat", "min_effective_sample_size", "divergences"]].copy()
intervals["simulation_row"] = intervals["simulation_row"].astype(int)
status["simulation_row"] = status["simulation_row"].astype(int)
intervals = intervals.merge(status, on="simulation_row", how="left", validate="many_to_one")
intervals["truth_inside_90pct_interval"] = intervals["truth_inside_90pct_interval"].astype(bool)

coverage_rows = []
for group_name, subset in [("all_run_cases", intervals),
                           ("screen_pass_only", intervals[intervals["status"] == "screen_pass"]),
                           ("needs_review_or_failed", intervals[intervals["status"] != "screen_pass"] )]:
    if subset.empty:
        continue
    for parameter, group in subset.groupby("parameter"):
        coverage_rows.append({"subset": group_name, "parameter": parameter,
            "case_parameter_intervals": int(len(group)),
            "truths_inside_90pct_interval": int(group["truth_inside_90pct_interval"].sum()),
            "descriptive_fraction": float(group["truth_inside_90pct_interval"].mean())})
coverage = pd.DataFrame(coverage_rows)

case_coverage = (intervals.groupby(["simulation_row", "status"], as_index=False)
    .agg(parameters_with_interval=("parameter", "nunique"),
         truths_covered=("truth_inside_90pct_interval", "sum")))
case_coverage["coverage_fraction_descriptive"] = case_coverage["truths_covered"] / case_coverage["parameters_with_interval"]
triage = nuts.merge(map_cases, on="simulation_row", how="left", suffixes=("_nuts", "_map"))
triage = triage[(triage["status"] != "screen_pass") | (~triage["period_recovered_within_tolerance"].astype(bool))].copy()

aliases["simulation_row"] = aliases["simulation_row"].astype(int)
map_cases["simulation_row"] = map_cases["simulation_row"].astype(int)
truth_period = map_cases.set_index("simulation_row")["known_period_days_for_evaluation"].to_dict()
aliases["delta_bic_from_best"] = aliases["bic"] - aliases.groupby("simulation_row")["bic"].transform("min")
aliases["truth_period_for_evaluation_only"] = aliases["simulation_row"].map(truth_period)
aliases["evaluation_tolerance_days"] = np.maximum(0.01, 0.002*aliases["truth_period_for_evaluation_only"])
aliases["mode_matches_truth_evaluation_only"] = (aliases["fit_period_days"]-aliases["truth_period_for_evaluation_only"]).abs() <= aliases["evaluation_tolerance_days"]
aliases["within_delta_bic_6"] = aliases["delta_bic_from_best"] <= 6.0
triage_rows = set(triage["simulation_row"].astype(int))
alias_triage = aliases[aliases["simulation_row"].isin(triage_rows)].copy()
alias_triage["best_bic_candidate"] = alias_triage["delta_bic_from_best"] <= 1e-9

coverage.to_csv(OUT/"day9_8_coverage_by_sampler_status.csv", index=False)
case_coverage.to_csv(OUT/"day9_8_case_interval_recovery.csv", index=False)
triage.to_csv(OUT/"day9_8_case_triage.csv", index=False)
alias_triage.sort_values(["simulation_row", "delta_bic_from_best"]).to_csv(OUT/"day9_8_alias_alternatives_for_triage.csv", index=False)

summary = {"created_utc": datetime.now(timezone.utc).isoformat(),
    "nuts_cases": int(len(nuts)),
    "screen_pass_cases": int(nuts["status"].eq("screen_pass").sum()),
    "needs_review_or_failed_cases": int(nuts["status"].ne("screen_pass").sum()),
    "map_period_recovered_cases": int(map_cases["period_recovered_within_tolerance"].astype(bool).sum()),
    "map_case_count": int(len(map_cases)),
    "cases_requiring_triage": sorted(int(x) for x in triage_rows),
    "interpretation": ["Coverage among screen-pass cases is still descriptive because the set has only 12 fixed stress-test cases.",
        "Nonconverged-chain intervals are listed separately and should not be used as validated posterior coverage.",
        "Truth-match flags are evaluation-only; candidate selection is by BIC, not by truth.",
        "Local-mode NUTS does not marginalize over alternative periods."],
    "outputs": ["day9_8_coverage_by_sampler_status.csv", "day9_8_case_interval_recovery.csv",
        "day9_8_case_triage.csv", "day9_8_alias_alternatives_for_triage.csv"]}
(OUT/"day9_8_audit_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("NUTS cases:", summary["nuts_cases"], "| screen pass:", summary["screen_pass_cases"],
      "| needs review/failed:", summary["needs_review_or_failed_cases"])
print("MAP period modes recovered:", summary["map_period_recovered_cases"], "/", summary["map_case_count"])
print("Coverage split by sampler status:")
print(coverage.to_string(index=False))
print("Cases for triage:", summary["cases_requiring_triage"])
print("Alternative BIC modes (inspect delta_bic_from_best and truth flag for evaluation only):")
print(alias_triage.sort_values(["simulation_row", "delta_bic_from_best"]).to_string(index=False))
print(json.dumps(summary, indent=2))
print("Outputs:", OUT)